In [ ]:
# Module 6 Notes - Tfds
# 1 | Tensorflow + tfds Setup

# tensorflow datasets gives you ready-made datasets with:
    # consistent structure
    # built-in splits
    # automatic downloading
    # (image, label) pairs
    # metadata (info object)


# Key imports
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.Keras import layers, Model

# Reproducibility

AUTOTUNE = tf.data.AUTOTUNE
SEED = 13
tf.keras.utils.set_random_seed(SEED)

# Artifacts Folder (save model)

BASE_DIR = pathlib.Path("artifacts_test")
(BASE_DIR/"checkpoints_test").mkdir(parents=True, exist_ok=True)
(BASE_DIR/"saved_models").mkdir(parents=True, exist_ok=True)

#allows me to save my best model weights & final models


In [ ]:
# 2 | Loading CIFAR-10 with TFDS

ds_all = tfds.load('cifar10', splits=['train', 'test'], as_supervised=True)
ds_train_raw, ds_testraw = ds_all

# why as_supervised=True? It gives (Image, Label) instead of dicts
# Because CIFAR-10 has no built-in val set:
VAL_FRACTION = 0.1
ds_train_raw = ds_train_raw.shuffle(10_000, seed=SEED, reshuffle_each_iteration=False) 
ds_val_raw = ds_train_raw.take(val_count)
ds_train_raw = ds_trainj_raw.skip(val_count)

In [ ]:
# 3 | Preprocessing Function

# Every example must be normalized, resized, and label cast to class int

def preprocess_cifar(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, tf.cast(label, tf.int32)


In [ ]:
# 4 | Building Efficient tf.data Pipelines

# main pipeline operations:
    # map() - Preprocessing
        # parallel, fast
    # batch() - Group samples
        # controls memory
    # prefatch() - Overlap CPU + GPU
        # makes GPU not wait

ds_train = (ds_train_raw
            .map(preprocess_cifar, AUTOTUNE)
            .batch(BATCH)
            .prefatch(AUTOTUNE))

# This pipeline is the backbone of TFDS usage



In [ ]:
# 5 | Building a Small CNN for CIFAR-10

# Three convolution blocks
 # used 32 filters, 64 filters, and 128 filters

# Each block:
x = layers.Conv2D(...)(inputs)
x = layers.Conv2D(...)(x)
x = layers.MaxPool2D()(x)

# Classification head

x = layers.Flatten()(conv_output)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs  = layers.Dense(10, activaiton='softmax')(x)

# We dropout so overfitting can be prevented

In [ ]:
# 6 | Model training + Checkpoints

# Compile

model.compile(optimizer='adam',
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])

# Callbacks:
    # Modelcheckpoints: Saves best val accuracy
    # EarlyStopping: stops when val plateaus

In [ ]:
# 7 | Transfer Learning - Cats & Dogs
 # this is the real purpose of AS6, it teaches three levels of transfer learning

# 7.1 | Steps

    # Load your best CIFAR model
    # Remove classification head
    # Feed cats/dogs images into the conv layers
    # Flatten outputs
    # Train a new dense classifier on these features

# This works because CIFAR filters detect edges, shapes, and textures, which are 
# enough for cat/dog separation

# 7.2 | Frozen Conv Base + Augmentation (End-to-End Transfer)

data_augmentation = keras.Sequential([
    RandomFlip, RandomRotation, RandomZoom
])

# pipeline:
 # Dataset - augmentation - conv base (frozen) - new classifier

 # this trains end-to-end by preserving CIFAR filters

# 7.3 | VGG16 Pretrained Transfer Learning
 # Breaking into ImageNet-level transfer

# Steps

    # Load VGG16 include_top=False
    # Resize images to 224x224
    # Apply correct VGG preprocessing (center mean subtraction)
    # Freeze entire conv base
        # Add GlobalAveragePooling2D
        # Dense - dropout - Dense
    # Train
    # Unfreeze last few layers
    # Fine-tune at low learning rate

# We use VGG16 because it has learned general visual features from 1.2M ImageNet images

# This gives a much stronger representation than CIFAR-10 conv base


In [ ]:
# 8 | Fine Tuning

# After freezing:

vgg_base.trainable = True
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

    # this unfreezes ONLY the last 4 layers

    # Why do we unfreeze only a few?
        # unfreezing everything leads to overfitting
        # small LR (learning rate ) prevents fatal forgetting
        # shallow layers store general features
        # deep layers store class-specific features
    

In [ ]:
# 9 | Slowness cause

# large CNNs
# tens of thousands of images
# with many parallel CPU steps
# on a CPU only
# GPU is faster